In [ ]:
# ================================================================
# TRUE DIC — One-Step Conditional Likelihood
# IID / Weekly BYM / Weekly BYM + Cov
# FULL DATA VERSION
# 10 chains pooled, thin = 15
# ================================================================

import numpy as np
import pyreadr
import pickle
import pandas as pd
from tqdm import tqdm
from pathlib import Path

# ------------------------------------------------
# Config
# ------------------------------------------------
BASE_DIR = Path(r"D:\77\Research\temp\snow")

period = 52
THIN = 15
N_CHAINS = 10

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# Load FULL data
# ================================================================
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
T = TT - 1

print("Using FULL S =", S, "TT =", TT)

# ================================================================
# Time covariates
# ================================================================
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

cos_all = np.cos(2 * np.pi * np.arange(1, TT) / period)
sin_all = np.sin(2 * np.pi * np.arange(1, TT) / period)
trend_all = t_scaled[:-1]

# ================================================================
# Covariates for +Cov model (FULL, aligned with full-data fitting)
# ================================================================
snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

lat = coords[:, 1]
lat = (lat - lat.mean()) / lat.std()

elev_raw = pd.read_csv(BASE_DIR / "curr_elev.csv").iloc[:, 3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR / "nnbs_elev.csv", sep="\t").iloc[:, 2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all = np.zeros(S)
elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

assert len(elev_raw) == mask.sum()
assert len(nnbs_elev) == len(no_nbs)
assert temp_scaled.shape[0] == S
assert temp_scaled.shape[1] == TT

# ================================================================
# Helper likelihood
# ================================================================
def one_step_ll(p01_list, p10_list):
    ll = 0.0

    for t in range(T):
        y_prev = y[:, t]
        y_next = y[:, t + 1]

        prob = np.where(
            y_prev == 0,
            np.where(y_next == 1, p01_list[t], 1 - p01_list[t]),
            np.where(y_next == 0, p10_list[t], 1 - p10_list[t])
        )

        ll += np.sum(np.log(prob + 1e-12))

    return ll

# ================================================================
# 1️⃣ IID
# ================================================================
print("\n===== IID TRUE DIC =====")

theta01_list = []
theta10_list = []

for c in range(N_CHAINS):
    d01 = np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")
    d10 = np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")

    key01 = d01.files[0]
    key10 = d10.files[0]

    theta01_list.append(d01[key01][:, ::THIN])
    theta10_list.append(d10[key10][:, ::THIN])

theta01 = np.concatenate(theta01_list, axis=1)
theta10 = np.concatenate(theta10_list, axis=1)

assert theta01.shape[0] == 4 * S
assert theta10.shape[0] == 4 * S

N_SAMPLE = theta01.shape[1]
print("IID posterior samples =", N_SAMPLE)

def compute_iid_ll(th01, th10):
    p01_list = []
    p10_list = []

    for t in range(T):
        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        phi01 = th01[0] + th01[1] * cos_t + th01[2] * sin_t + th01[3] * trend
        phi10 = th10[0] + th10[1] * cos_t + th10[2] * sin_t + th10[3] * trend

        p01_list.append(1 / (1 + np.exp(-phi01)))
        p10_list.append(1 / (1 + np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE), desc="IID posterior"):
    D_vals[m] = -2 * compute_iid_ll(
        theta01[:, m].reshape(4, S),
        theta10[:, m].reshape(4, S)
    )

D_bar = D_vals.mean()

D_hat = -2 * compute_iid_ll(
    theta01.mean(axis=1).reshape(4, S),
    theta10.mean(axis=1).reshape(4, S)
)

DIC_IID = 2 * D_bar - D_hat

print("IID D_bar:", D_bar)
print("IID D_hat:", D_hat)
print("IID DIC  :", DIC_IID)

# ================================================================
# 2️⃣ Weekly BYM
# ================================================================
print("\n===== Weekly BYM TRUE DIC =====")

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

K_total = 8

assert eta01.shape[0] == K_total * S
assert eta10.shape[0] == K_total * S
assert tau01.shape[0] == K_total * period
assert tau10.shape[0] == K_total * period

N_SAMPLE = eta01.shape[1]
print("Weekly BYM posterior samples =", N_SAMPLE)

def compute_weekly_ll(e01, t01, e10, t10):
    e01_sp = e01.reshape(K_total, S)
    e10_sp = e10.reshape(K_total, S)

    t01_mat = t01.reshape(K_total, period)
    t10_mat = t10.reshape(K_total, period)

    p01_list = []
    p10_list = []

    for t in range(T):
        week = t % period

        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        cov_vec = np.array([1, 1, cos_t, cos_t, sin_t, sin_t, trend, trend])

        phi01 = np.sum(cov_vec[:, None] * e01_sp * t01_mat[:, week][:, None], axis=0)
        phi10 = np.sum(cov_vec[:, None] * e10_sp * t10_mat[:, week][:, None], axis=0)

        p01_list.append(1 / (1 + np.exp(-phi01)))
        p10_list.append(1 / (1 + np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE), desc="Weekly BYM posterior"):
    D_vals[m] = -2 * compute_weekly_ll(
        eta01[:, m], tau01[:, m],
        eta10[:, m], tau10[:, m]
    )

D_bar = D_vals.mean()

D_hat = -2 * compute_weekly_ll(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
)

DIC_weekly = 2 * D_bar - D_hat

print("Weekly BYM D_bar:", D_bar)
print("Weekly BYM D_hat:", D_hat)
print("Weekly BYM DIC  :", DIC_weekly)

# ================================================================
# 3️⃣ Weekly BYM + Cov
# ================================================================
print("\n===== Weekly BYM + Cov TRUE DIC =====")

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

assert tau01.shape[0] == K_total * period
assert tau10.shape[0] == K_total * period
assert eta01.shape[0] == K_total * S + 3
assert eta10.shape[0] == K_total * S + 3

N_SAMPLE = eta01.shape[1]
print("Weekly BYM + Cov posterior samples =", N_SAMPLE)

def compute_weekly_cov_ll(e01, t01, e10, t10):
    e01_sp = e01[:K_total * S].reshape(K_total, S)
    gamma01 = e01[K_total * S:]

    e10_sp = e10[:K_total * S].reshape(K_total, S)
    gamma10 = e10[K_total * S:]

    t01_mat = t01.reshape(K_total, period)
    t10_mat = t10.reshape(K_total, period)

    p01_list = []
    p10_list = []

    for t in range(T):
        week = t % period

        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        cov_vec = np.array([1, 1, cos_t, cos_t, sin_t, sin_t, trend, trend])

        phi01_sp = np.sum(cov_vec[:, None] * e01_sp * t01_mat[:, week][:, None], axis=0)
        phi10_sp = np.sum(cov_vec[:, None] * e10_sp * t10_mat[:, week][:, None], axis=0)

        phi01 = (
            phi01_sp
            + trend * lat * gamma01[0]
            + trend * elev * gamma01[1]
            + trend * temp_scaled[:, t] * gamma01[2]
        )

        phi10 = (
            phi10_sp
            + trend * lat * gamma10[0]
            + trend * elev * gamma10[1]
            + trend * temp_scaled[:, t] * gamma10[2]
        )

        p01_list.append(1 / (1 + np.exp(-phi01)))
        p10_list.append(1 / (1 + np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE), desc="Weekly BYM + Cov posterior"):
    D_vals[m] = -2 * compute_weekly_cov_ll(
        eta01[:, m], tau01[:, m],
        eta10[:, m], tau10[:, m]
    )

D_bar = D_vals.mean()

D_hat = -2 * compute_weekly_cov_ll(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
)

DIC_weekly_cov = 2 * D_bar - D_hat

print("Weekly BYM + Cov D_bar:", D_bar)
print("Weekly BYM + Cov D_hat:", D_hat)
print("Weekly BYM + Cov DIC  :", DIC_weekly_cov)

# ================================================================
# Final
# ================================================================
print("\n==============================")
print("IID DIC              :", DIC_IID)
print("Weekly BYM DIC       :", DIC_weekly)
print("Weekly BYM + Cov DIC :", DIC_weekly_cov)
print("==============================")

Using FULL S = 1618 TT = 2704

===== IID TRUE DIC =====
IID posterior samples = 3340


IID posterior: 100%|██████████| 3340/3340 [22:47<00:00,  2.44it/s]


IID D_bar: 1264311.104459618
IID D_hat: 1251693.3133693694
IID DIC  : 1276928.8955498666

===== Weekly BYM TRUE DIC =====
Weekly BYM posterior samples = 3340


Weekly BYM posterior: 100%|██████████| 3340/3340 [41:11<00:00,  1.35it/s]


Weekly BYM D_bar: 1225200.9766956982
Weekly BYM D_hat: 1246015.2079041663
Weekly BYM DIC  : 1204386.7454872301

===== Weekly BYM + Cov TRUE DIC =====
Weekly BYM + Cov posterior samples = 3340


Weekly BYM + Cov posterior: 100%|██████████| 3340/3340 [34:25<00:00,  1.62it/s]


Weekly BYM + Cov D_bar: nan
Weekly BYM + Cov D_hat: nan
Weekly BYM + Cov DIC  : nan

IID DIC              : 1276928.8955498666
Weekly BYM DIC       : 1204386.7454872301
Weekly BYM + Cov DIC : nan


In [20]:
print("\n===== Weekly BYM + Cov TRUE DIC =====")

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

assert tau01.shape[0] == K_total * period
assert tau10.shape[0] == K_total * period
assert eta01.shape[0] == K_total * S + 3
assert eta10.shape[0] == K_total * S + 3

N_SAMPLE = eta01.shape[1]
print("Weekly BYM + Cov posterior samples =", N_SAMPLE)

def compute_weekly_cov_ll(e01, t01, e10, t10):
    e01_sp = e01[:K_total * S].reshape(K_total, S)
    gamma01 = e01[K_total * S:]

    e10_sp = e10[:K_total * S].reshape(K_total, S)
    gamma10 = e10[K_total * S:]

    t01_mat = t01.reshape(K_total, period)
    t10_mat = t10.reshape(K_total, period)

    p01_list = []
    p10_list = []

    for t in range(T):
        week = t % period

        cos_t = cos_all[t]
        sin_t = sin_all[t]
        trend = trend_all[t]

        cov_vec = np.array([1, 1, cos_t, cos_t, sin_t, sin_t, trend, trend])

        phi01_sp = np.sum(cov_vec[:, None] * e01_sp * t01_mat[:, week][:, None], axis=0)
        phi10_sp = np.sum(cov_vec[:, None] * e10_sp * t10_mat[:, week][:, None], axis=0)

        phi01 = (
            phi01_sp
            + trend * lat * gamma01[0]
            + trend * elev * gamma01[1]
            + trend * temp_scaled[:, t] * gamma01[2]
        )

        phi10 = (
            phi10_sp
            + trend * lat * gamma10[0]
            + trend * elev * gamma10[1]
            + trend * temp_scaled[:, t] * gamma10[2]
        )

        p01_list.append(1 / (1 + np.exp(-phi01)))
        p10_list.append(1 / (1 + np.exp(-phi10)))

    return one_step_ll(p01_list, p10_list)

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE), desc="Weekly BYM + Cov posterior"):

    ll_val = compute_weekly_cov_ll(
        eta01[:, m], tau01[:, m],
        eta10[:, m], tau10[:, m]
    )

    D_val = -2 * ll_val
    D_vals[m] = D_val

D_bar = D_vals.mean()

D_hat = -2 * compute_weekly_cov_ll(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
)

DIC_weekly_cov = 2 * D_bar - D_hat

print("Weekly BYM + Cov D_bar:", D_bar)
print("Weekly BYM + Cov D_hat:", D_hat)
print("Weekly BYM + Cov DIC  :", DIC_weekly_cov)


===== Weekly BYM + Cov TRUE DIC =====
Weekly BYM + Cov posterior samples = 3340


Weekly BYM + Cov posterior: 100%|██████████| 3340/3340 [42:23<00:00,  1.31it/s]


Weekly BYM + Cov D_bar: 1224104.7162641473
Weekly BYM + Cov D_hat: 1247710.956544392
Weekly BYM + Cov DIC  : 1200498.4759839026


In [22]:
print("\n==============================")
print("IID DIC              :", DIC_IID)
print("Weekly BYM DIC       :", DIC_weekly)
print("Weekly BYM + Cov DIC :", DIC_weekly_cov)
print("==============================")


IID DIC              : 1276928.8955498666
Weekly BYM DIC       : 1204386.7454872301
Weekly BYM + Cov DIC : 1200498.4759839026


In [1]:
# ================================================================
# TRUE WAIC — One-Step Conditional Likelihood
# IID / Weekly BYM / Weekly BYM + Cov
# FULL DATA VERSION
# 10 chains pooled, thin = 15
# ================================================================

import numpy as np
import pyreadr
import pickle
import pandas as pd
from scipy.special import logsumexp
from tqdm import tqdm
from pathlib import Path

BASE_DIR = Path(r"D:\77\Research\temp\snow")

period = 52
THIN = 15
N_CHAINS = 10

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# Load FULL data
# ================================================================
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
T = TT - 1

print("Using FULL S =", S, "TT =", TT)

# ================================================================
# Time covariates
# ================================================================
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

cos_all = np.cos(2 * np.pi * np.arange(1, TT) / period)
sin_all = np.sin(2 * np.pi * np.arange(1, TT) / period)
trend_all = t_scaled[:-1]

# ================================================================
# Covariates for +Cov model (FULL, aligned with fitting)
# ================================================================
snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

lat = coords[:, 1]
lat = (lat - lat.mean()) / lat.std()

elev_raw = pd.read_csv(BASE_DIR / "curr_elev.csv").iloc[:, 3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR / "nnbs_elev.csv", sep="\t").iloc[:, 2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all = np.zeros(S)
elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

assert len(elev_raw) == mask.sum()
assert len(nnbs_elev) == len(no_nbs)
assert temp_scaled.shape[0] == S
assert temp_scaled.shape[1] == TT

# ================================================================
# Transition data
# ================================================================
y_prev = y[:, :-1]
y_next = y[:, 1:]

# ================================================================
# WAIC accumulator
# ================================================================
def update_waic(loglik, lppd_acc, p_acc):
    lppd_acc += np.sum(logsumexp(loglik, axis=1) - np.log(loglik.shape[1]))
    p_acc += np.sum(np.var(loglik, axis=1))
    return lppd_acc, p_acc

# ================================================================
# 1️⃣ IID WAIC
# ================================================================
print("\n===== IID WAIC =====")

theta01_list = []
theta10_list = []

for c in range(N_CHAINS):
    d01 = np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")
    d10 = np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")

    key01 = d01.files[0]
    key10 = d10.files[0]

    theta01_list.append(d01[key01][:, ::THIN])
    theta10_list.append(d10[key10][:, ::THIN])

theta01 = np.concatenate(theta01_list, axis=1)
theta10 = np.concatenate(theta10_list, axis=1)

assert theta01.shape[0] == 4 * S
assert theta10.shape[0] == 4 * S

M = theta01.shape[1]
print("IID posterior samples =", M)

theta01 = theta01.reshape(4, S, M)
theta10 = theta10.reshape(4, S, M)

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(T), desc="IID WAIC"):
    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    phi01 = theta01[0] + theta01[1] * cos_t + theta01[2] * sin_t + theta01[3] * trend
    phi10 = theta10[0] + theta10[1] * cos_t + theta10[2] * sin_t + theta10[3] * trend

    p01 = 1 / (1 + np.exp(-phi01))
    p10 = 1 / (1 + np.exp(-phi10))

    prob = np.where(
        y_prev[:, t][:, None] == 0,
        np.where(y_next[:, t][:, None] == 1, p01, 1 - p01),
        np.where(y_next[:, t][:, None] == 0, p10, 1 - p10)
    )

    loglik = np.log(prob + 1e-12)
    lppd, p_waic = update_waic(loglik, lppd, p_waic)

WAIC_IID = -2 * (lppd - p_waic)

print("IID lppd   :", lppd)
print("IID p_waic :", p_waic)
print("IID WAIC   :", WAIC_IID)

# ================================================================
# 2️⃣ Weekly BYM WAIC
# ================================================================
print("\n===== Weekly BYM WAIC =====")

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

M = eta01.shape[1]
K_total = 8

assert eta01.shape[0] == K_total * S
assert eta10.shape[0] == K_total * S
assert tau01.shape[0] == K_total * period
assert tau10.shape[0] == K_total * period

print("Weekly BYM posterior samples =", M)

eta01 = eta01.reshape(K_total, S, M)
eta10 = eta10.reshape(K_total, S, M)

tau01 = tau01.reshape(K_total, period, M)
tau10 = tau10.reshape(K_total, period, M)

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(T), desc="Weekly BYM WAIC"):
    week = t % period
    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    cov_vec = np.array([1, 1, cos_t, cos_t, sin_t, sin_t, trend, trend])

    phi01 = np.zeros((S, M))
    phi10 = np.zeros((S, M))

    for k in range(K_total):
        phi01 += cov_vec[k] * eta01[k] * tau01[k, week]
        phi10 += cov_vec[k] * eta10[k] * tau10[k, week]

    p01 = 1 / (1 + np.exp(-phi01))
    p10 = 1 / (1 + np.exp(-phi10))

    prob = np.where(
        y_prev[:, t][:, None] == 0,
        np.where(y_next[:, t][:, None] == 1, p01, 1 - p01),
        np.where(y_next[:, t][:, None] == 0, p10, 1 - p10)
    )

    loglik = np.log(prob + 1e-12)
    lppd, p_waic = update_waic(loglik, lppd, p_waic)

WAIC_weekly = -2 * (lppd - p_waic)

print("Weekly BYM lppd   :", lppd)
print("Weekly BYM p_waic :", p_waic)
print("Weekly BYM WAIC   :", WAIC_weekly)

# ================================================================
# 3️⃣ Weekly BYM + Cov WAIC
# ================================================================
print("\n===== Weekly BYM + Cov WAIC =====")

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_cov_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

M = eta01.shape[1]

assert eta01.shape[0] == K_total * S + 3
assert eta10.shape[0] == K_total * S + 3
assert tau01.shape[0] == K_total * period
assert tau10.shape[0] == K_total * period

print("Weekly BYM + Cov posterior samples =", M)

gamma01 = eta01[K_total * S:, :]
gamma10 = eta10[K_total * S:, :]

eta01 = eta01[:K_total * S].reshape(K_total, S, M)
eta10 = eta10[:K_total * S].reshape(K_total, S, M)

tau01 = tau01.reshape(K_total, period, M)
tau10 = tau10.reshape(K_total, period, M)

lppd = 0.0
p_waic = 0.0

for t in tqdm(range(T), desc="Weekly BYM + Cov WAIC"):
    week = t % period
    cos_t = cos_all[t]
    sin_t = sin_all[t]
    trend = trend_all[t]

    cov_vec = np.array([1, 1, cos_t, cos_t, sin_t, sin_t, trend, trend])

    phi01_sp = np.zeros((S, M))
    phi10_sp = np.zeros((S, M))

    for k in range(K_total):
        phi01_sp += cov_vec[k] * eta01[k] * tau01[k, week]
        phi10_sp += cov_vec[k] * eta10[k] * tau10[k, week]

    phi01 = (
        phi01_sp
        + trend * lat[:, None] * gamma01[0]
        + trend * elev[:, None] * gamma01[1]
        + trend * temp_scaled[:, t][:, None] * gamma01[2]
    )

    phi10 = (
        phi10_sp
        + trend * lat[:, None] * gamma10[0]
        + trend * elev[:, None] * gamma10[1]
        + trend * temp_scaled[:, t][:, None] * gamma10[2]
    )

    p01 = 1 / (1 + np.exp(-phi01))
    p10 = 1 / (1 + np.exp(-phi10))

    prob = np.where(
        y_prev[:, t][:, None] == 0,
        np.where(y_next[:, t][:, None] == 1, p01, 1 - p01),
        np.where(y_next[:, t][:, None] == 0, p10, 1 - p10)
    )

    loglik = np.log(prob + 1e-12)
    lppd, p_waic = update_waic(loglik, lppd, p_waic)

WAIC_weekly_cov = -2 * (lppd - p_waic)

print("Weekly BYM + Cov lppd   :", lppd)
print("Weekly BYM + Cov p_waic :", p_waic)
print("Weekly BYM + Cov WAIC   :", WAIC_weekly_cov)

# ================================================================
# Final
# ================================================================
print("\n==============================")
print("IID WAIC              :", WAIC_IID)
print("Weekly BYM WAIC       :", WAIC_weekly)
print("Weekly BYM + Cov WAIC :", WAIC_weekly_cov)
print("==============================")

Using FULL S = 1618 TT = 2704

===== IID WAIC =====
IID posterior samples = 3340


IID WAIC: 100%|██████████| 2703/2703 [16:35<00:00,  2.71it/s]


IID lppd   : -625770.470828119
IID p_waic : 13189.048504743334
IID WAIC   : 1277919.0386657247

===== Weekly BYM WAIC =====
Weekly BYM posterior samples = 3340


Weekly BYM WAIC: 100%|██████████| 2703/2703 [27:33<00:00,  1.64it/s]


Weekly BYM lppd   : -605933.061563581
Weekly BYM p_waic : 13641.754011296314
Weekly BYM WAIC   : 1239149.6311497546

===== Weekly BYM + Cov WAIC =====
Weekly BYM + Cov posterior samples = 3340


Weekly BYM + Cov WAIC: 100%|██████████| 2703/2703 [42:33<00:00,  1.06it/s]

Weekly BYM + Cov lppd   : -605405.7366812627
Weekly BYM + Cov p_waic : 13600.549516211819
Weekly BYM + Cov WAIC   : 1238012.5723949492

IID WAIC              : 1277919.0386657247
Weekly BYM WAIC       : 1239149.6311497546
Weekly BYM + Cov WAIC : 1238012.5723949492


In [1]:
# Test for t^2 WAIC and DIC

# ================================================================
# DIC + WAIC for Weekly BYM + Cov (t^2 version, SINGLE CHAIN)
# ================================================================

import numpy as np
import pyreadr
import pickle
import pandas as pd
from tqdm import tqdm
from pathlib import Path

BASE_DIR = Path(r"D:\77\Research\temp\snow")

period = 52
THIN = 1  

# ================================================================
# LOAD DATA
# ================================================================
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
T = TT - 1

# ================================================================
# TIME TERMS (t and t^2 !!!)
# ================================================================
t_full = np.arange(1, TT + 1)

t_scaled = (t_full - t_full.mean()) / t_full.std()
t2_full = t_full**2
t2_scaled = (t2_full - t2_full.mean()) / t2_full.std()

cos_all = np.cos(2 * np.pi * np.arange(1, TT) / period)
sin_all = np.sin(2 * np.pi * np.arange(1, TT) / period)

trend_all = t_scaled[:-1]
trend2_all = t2_scaled[:-1]

# ================================================================
# COVARIATES (same as model)
# ================================================================
snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

lat = coords[:, 1]
lat = (lat - lat.mean()) / lat.std()

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

elev_raw = pd.read_csv(BASE_DIR / "curr_elev.csv").iloc[:, 3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR / "nnbs_elev.csv", sep="\t").iloc[:, 2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all = np.zeros(S)
elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

# ================================================================
# LOAD SINGLE CHAIN
# ================================================================
with open(BASE_DIR / "p01_weekly_covt2_chain0.pkl", "rb") as f:
    d = pickle.load(f)
eta01 = d["eta"][:, ::THIN]
tau01 = d["tau"][:, ::THIN]

with open(BASE_DIR / "p10_weekly_covt2_chain0.pkl", "rb") as f:
    d = pickle.load(f)
eta10 = d["eta"][:, ::THIN]
tau10 = d["tau"][:, ::THIN]

K_base = 5
K_total = 2 * K_base

N_SAMPLE = eta01.shape[1]
print("Samples =", N_SAMPLE)

# ================================================================
# LOG-LIK PER TIME (for WAIC)
# ================================================================
def compute_loglik_matrix(e01, t01, e10, t10):

    e01_sp = e01[:K_total*S].reshape(K_total, S)
    gamma01 = e01[K_total*S:]

    e10_sp = e10[:K_total*S].reshape(K_total, S)
    gamma10 = e10[K_total*S:]

    t01_mat = t01.reshape(K_total, period)
    t10_mat = t10.reshape(K_total, period)

    loglik = np.zeros((T, S))

    for t in range(T):
        week = t % period

        cov_vec = np.array([
            1,1,
            cos_all[t],cos_all[t],
            sin_all[t],sin_all[t],
            trend_all[t],trend_all[t],
            trend2_all[t],trend2_all[t]
        ])

        phi01_sp = np.sum(cov_vec[:, None] * e01_sp * t01_mat[:, week][:, None], axis=0)
        phi10_sp = np.sum(cov_vec[:, None] * e10_sp * t10_mat[:, week][:, None], axis=0)

        phi01 = (
            phi01_sp
            + trend_all[t] * lat * gamma01[0]
            + trend_all[t] * elev * gamma01[1]
            + trend_all[t] * temp_scaled[:, t] * gamma01[2]
        )

        phi10 = (
            phi10_sp
            + trend_all[t] * lat * gamma10[0]
            + trend_all[t] * elev * gamma10[1]
            + trend_all[t] * temp_scaled[:, t] * gamma10[2]
        )

        p01 = 1 / (1 + np.exp(-phi01))
        p10 = 1 / (1 + np.exp(-phi10))

        y_prev = y[:, t]
        y_next = y[:, t+1]

        prob = np.where(
            y_prev == 0,
            np.where(y_next == 1, p01, 1 - p01),
            np.where(y_next == 0, p10, 1 - p10)
        )

        loglik[t] = np.log(prob + 1e-12)

    return loglik


# ================================================================
# DIC
# ================================================================
print("\n===== DIC =====")

D_vals = np.zeros(N_SAMPLE)

for m in tqdm(range(N_SAMPLE)):
    ll = compute_loglik_matrix(
        eta01[:, m], tau01[:, m],
        eta10[:, m], tau10[:, m]
    ).sum()

    D_vals[m] = -2 * ll

D_bar = D_vals.mean()

# posterior mean
ll_hat = compute_loglik_matrix(
    eta01.mean(axis=1), tau01.mean(axis=1),
    eta10.mean(axis=1), tau10.mean(axis=1)
).sum()

D_hat = -2 * ll_hat

DIC = 2 * D_bar - D_hat

print("D_bar:", D_bar)
print("D_hat:", D_hat)
print("DIC  :", DIC)


# ================================================================
# WAIC
# ================================================================
print("\n===== WAIC =====")

loglik_all = np.zeros((N_SAMPLE, T, S))

for m in tqdm(range(N_SAMPLE)):
    loglik_all[m] = compute_loglik_matrix(
        eta01[:, m], tau01[:, m],
        eta10[:, m], tau10[:, m]
    )

# lppd
lppd = np.sum(
    np.log(np.mean(np.exp(loglik_all), axis=0) + 1e-12)
)

# p_waic
p_waic = np.sum(np.var(loglik_all, axis=0))

WAIC = -2 * (lppd - p_waic)

print("lppd :", lppd)
print("p_waic:", p_waic)
print("WAIC :", WAIC)


# ================================================================
# FINAL
# ================================================================
print("\n==============================")
print("DIC  :", DIC)
print("WAIC :", WAIC)
print("==============================")


Samples = 1000

===== DIC =====


  5%|▌         | 54/1000 [00:32<09:30,  1.66it/s]C:\Users\Qi\AppData\Local\Temp\ipykernel_17308\1692122472.py:138: RuntimeWarning: overflow encountered in exp
  p10 = 1 / (1 + np.exp(-phi10))
100%|██████████| 1000/1000 [10:05<00:00,  1.65it/s]


D_bar: 1218432.9105644808
D_hat: 1205049.4750721357
DIC  : 1231816.346056826

===== WAIC =====


100%|██████████| 1000/1000 [12:20<00:00,  1.35it/s]


lppd : -601884.3164328401
p_waic: 15033.977066872918
WAIC : 1233836.586999426

DIC  : 1231816.346056826
WAIC : 1233836.586999426


In [ ]:
# ================================================================
# TRUE WAIC — Weekly BYM + Cov (t^2)
# ================================================================

import numpy as np
import pyreadr
import pickle
import pandas as pd
from scipy.special import logsumexp
from tqdm import tqdm
from pathlib import Path

BASE_DIR = Path(r"D:\77\Research\temp\snow")

period = 52
THIN = 1
N_CHAINS = 1   # 你现在是单 chain

# ================================================================
# LOAD DATA
# ================================================================
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
T = TT - 1

# ================================================================
# TIME TERMS
# ================================================================
t_full = np.arange(1, TT + 1)

t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)
t2_scaled = (t_full**2 - (t_full**2).mean()) / (t_full**2).std(ddof=0)

cos_all = np.cos(2*np.pi*np.arange(1, TT)/period)
sin_all = np.sin(2*np.pi*np.arange(1, TT)/period)

trend_all  = t_scaled[:-1]
trend2_all = t2_scaled[:-1]

# ================================================================
# COVARIATES
# ================================================================
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
temp = list(snow_temp.values())[0].iloc[:,2:].to_numpy()
temp_scaled = (temp - temp.mean()) / temp.std()

lat = coords[:,1]
lat = (lat - lat.mean()) / lat.std()

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR/"nnbs_elev.csv", sep="\t").iloc[:,2].to_numpy()

mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all = np.zeros(S)
elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

# ================================================================
# TRANSITIONS
# ================================================================
y_prev = y[:, :-1]
y_next = y[:, 1:]

# ================================================================
# LOAD POSTERIOR 
# ================================================================
eta01_list, tau01_list = [], []
eta10_list, tau10_list = [], []

for c in range(N_CHAINS):
    with open(BASE_DIR / f"p01_weekly_covt2_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::THIN])
    tau01_list.append(d["tau"][:, ::THIN])

    with open(BASE_DIR / f"p10_weekly_covt2_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::THIN])
    tau10_list.append(d["tau"][:, ::THIN])

eta01 = np.concatenate(eta01_list, axis=1)
tau01 = np.concatenate(tau01_list, axis=1)
eta10 = np.concatenate(eta10_list, axis=1)
tau10 = np.concatenate(tau10_list, axis=1)

M = eta01.shape[1]

# ================================================================
# MODEL DIM
# ================================================================
K_base = 5
K_total = 2 * K_base

gamma01 = eta01[K_total*S:]
gamma10 = eta10[K_total*S:]

eta01 = eta01[:K_total*S].reshape(K_total, S, M)
eta10 = eta10[:K_total*S].reshape(K_total, S, M)

tau01 = tau01.reshape(K_total, period, M)
tau10 = tau10.reshape(K_total, period, M)

# ================================================================
# WAIC
# ================================================================
lppd = 0.0
p_waic = 0.0

for t in tqdm(range(T), desc="WAIC t2"):

    week = t % period

    cov_vec = np.array([
        1,1,
        cos_all[t],cos_all[t],
        sin_all[t],sin_all[t],
        trend_all[t],trend_all[t],
        trend2_all[t],trend2_all[t]
    ])

    phi01_sp = np.zeros((S, M))
    phi10_sp = np.zeros((S, M))

    for k in range(K_total):
        phi01_sp += cov_vec[k] * eta01[k] * tau01[k, week]
        phi10_sp += cov_vec[k] * eta10[k] * tau10[k, week]

    phi01 = (
        phi01_sp
        + trend_all[t]*lat[:,None]*gamma01[0]
        + trend_all[t]*elev[:,None]*gamma01[1]
        + trend_all[t]*temp_scaled[:,t][:,None]*gamma01[2]
    )

    phi10 = (
        phi10_sp
        + trend_all[t]*lat[:,None]*gamma10[0]
        + trend_all[t]*elev[:,None]*gamma10[1]
        + trend_all[t]*temp_scaled[:,t][:,None]*gamma10[2]
    )

    p01 = 1/(1+np.exp(-phi01))
    p10 = 1/(1+np.exp(-phi10))

    prob = np.where(
        y_prev[:,t][:,None]==0,
        np.where(y_next[:,t][:,None]==1, p01, 1-p01),
        np.where(y_next[:,t][:,None]==0, p10, 1-p10)
    )

    loglik = np.log(prob + 1e-12)

    lppd += np.sum(logsumexp(loglik, axis=1) - np.log(M))
    p_waic += np.sum(np.var(loglik, axis=1))

WAIC = -2*(lppd - p_waic)

print("\n===== RESULT =====")
print("WAIC :", WAIC)

WAIC t2:  61%|██████    | 1652/2703 [06:27<03:55,  4.46it/s]C:\Users\Qi\AppData\Local\Temp\ipykernel_17308\3881517913.py:156: RuntimeWarning: overflow encountered in exp
  p10 = 1/(1+np.exp(-phi10))
WAIC t2: 100%|██████████| 2703/2703 [10:37<00:00,  4.24it/s]


===== RESULT =====
WAIC : 1233836.5870136651
